# 01 - Landing to Bronze | V-Energy Grid 360
**Objetivo:** Ingestão fiel e auditável dos 8 datasets operacionais brutos da V-Energy para tabelas Delta na camada Bronze (`Unity Catalog`), preservando 100% do conteúdo original como texto/estrutura bruta e adicionando metadados de rastreabilidade (`_ingestion_timestamp` e `_source_file`).

### Problemas de Engenharia Tratados nesta Camada:
1. **Resolução Dinâmica de Subpastas (`base_dados_1`, `base_dados_2`):** Localiza automaticamente cada arquivo ou diretório (`leituras`) dentro da árvore do Volume da Landing Zone, independentemente de terem sido subidos na raiz ou divididos em subpastas.
2. **Delimitadores heterogêneos (`,` vs `;`):** Conforme relatado pela Diretora de TI (*"O faturamento sai em CSV com ponto e vírgula"*), a leitura detecta automaticamente o separador real de cada arquivo CSV antes de instanciar o leitor do Spark.
3. **JSON Aninhado / Pretty-Printed (`atendimentos.json`):** Uso obrigatório de `multiLine=True` para evitar `_corrupt_record` em arquivos JSON formatados em múltiplas linhas ou arrays JSON.
4. **Preservação de Nulos no Excel (`inspecoes.xlsx`):** Conversão de `NaN`/`NaT` para `None` antes da tipagem para `StringType`, evitando que valores ausentes virem a string literal `"nan"`. As 3 abas são unificadas em **uma única tabela** (`tb_inspecoes`), com a aba de origem na coluna `_source_sheet`.
5. **Rastreabilidade de Arquivos Particionados (`leituras/*.csv`):** Captura da coluna oculta `_metadata.file_path` do Unity Catalog para saber exatamente de qual arquivo anual (`leituras_2022.csv` .. `leituras_2026.csv`) veio cada registro.
6. **Evolução/Sobrescrita Segura de Schema (`overwriteSchema=true`):** Garante idempotência total caso o notebook seja reexecutado.
7. **Validação de completude (arquivo × tabela):** Para cada CSV, o número de linhas do arquivo (menos os cabeçalhos) deve ser igual ao número de linhas gravadas. Se uma linha se perder ou quebrar na leitura, o notebook falha.

In [0]:
%pip install openpyxl -q

## 1. Parâmetros, Configuração e Isolamento de Schemas

In [0]:
import os
import re
import unicodedata
from pyspark.sql import DataFrame
from pyspark.sql.functions import current_timestamp, col, lit

# Widgets permitem orquestrar este notebook via Databricks Workflows (Jobs)
dbutils.widgets.text("catalog", "workspace", "Catálogo Unity Catalog")
dbutils.widgets.text("landing_path", "/Volumes/workspace/landing/v_energy", "Caminho do Volume Landing")
dbutils.widgets.text("schema_prefix", "", "Prefixo dos Schemas (ex: v_energy_ ou vazio)")

catalog = dbutils.widgets.get("catalog").strip()
landing_path = dbutils.widgets.get("landing_path").strip().rstrip("/")
schema_prefix = dbutils.widgets.get("schema_prefix").strip()

bronze_schema = f"{catalog}.{schema_prefix}bronze"
silver_schema = f"{catalog}.{schema_prefix}silver"
gold_schema   = f"{catalog}.{schema_prefix}gold"

# Criação garantida dos schemas no Unity Catalog
for schema in [bronze_schema, silver_schema, gold_schema]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

print(f"Catálogo configurado : {catalog}")
print(f"Volume Landing Zone  : {landing_path}")
print(f"Schemas ativos       : {bronze_schema} | {silver_schema} | {gold_schema}")

## 2. Funções Utilitárias de Engenharia e Auditoria Bronze

In [0]:
def resolve_landing_paths(base_dir: str, required_items: list) -> dict:
    """
    Percorre recursivamente o Volume da Landing Zone (incluindo subpastas como
    'base_dados_1' e 'base_dados_2') e retorna um dicionário {nome_item: caminho_absoluto}.
    """
    if not os.path.exists(base_dir):
        raise FileNotFoundError(
            f"CRÍTICO: O diretório base '{base_dir}' não existe. Verifique o Volume no Unity Catalog."
        )

    found = {}
    for root, dirs, files in os.walk(base_dir):
        for d in dirs:
            if d in required_items and d not in found:
                found[d] = os.path.join(root, d).replace("\\", "/")
        for f in files:
            if f in required_items and f not in found:
                found[f] = os.path.join(root, f).replace("\\", "/")

    missing = [item for item in required_items if item not in found]
    if missing:
        raise FileNotFoundError(
            f"CRÍTICO: Itens obrigatórios não encontrados em nenhuma subpasta de '{base_dir}': {missing}\n"
            f"Itens localizados: {found}"
        )
    return found


def sanitize_identifier(name: str) -> str:
    """
    Remove acentos, caracteres especiais e espaços de nomes de colunas ou abas
    para evitar DeltaAnalysisException no Delta Lake.
    """
    nfkd = unicodedata.normalize("NFKD", str(name).strip())
    ascii_str = nfkd.encode("ASCII", "ignore").decode("ASCII")
    clean = re.sub(r"[^a-zA-Z0-9_]", "_", ascii_str.lower())
    clean = re.sub(r"_+", "_", clean).strip("_")
    return clean or "col_unnamed"


def sanitize_columns(df: DataFrame) -> DataFrame:
    """Padroniza os nomes das colunas em snake_case mantendo os dados intactos."""
    seen = {}
    new_cols = []
    for c in df.columns:
        base = sanitize_identifier(c)
        if base in seen:
            seen[base] += 1
            new_cols.append(f"{base}_{seen[base]}")
        else:
            seen[base] = 0
            new_cols.append(base)
    return df.toDF(*new_cols)


def detect_csv_delimiter(file_path: str) -> str:
    """
    Inspeciona a primeira linha do arquivo no Volume para detectar automaticamente
    se o delimitador é ';' (ex: faturas.csv) ou ',' (padrão).
    """
    sample_path = file_path
    if os.path.isdir(file_path):
        csv_files = [
            os.path.join(file_path, f)
            for f in os.listdir(file_path)
            if f.lower().endswith(".csv")
        ]
        if not csv_files:
            raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em {file_path}")
        sample_path = sorted(csv_files)[0]

    with open(sample_path, "r", encoding="utf-8", errors="replace") as f:
        header_line = f.readline()

    semicolons = header_line.count(";")
    commas = header_line.count(",")
    pipes = header_line.count("|")

    if semicolons > commas and semicolons >= pipes:
        return ";"
    if pipes > commas and pipes > semicolons:
        return "|"
    return ","


def write_bronze_delta(df: DataFrame, full_table_name: str, source_col_expr=None) -> dict:
    """
    Adiciona colunas de metadados de auditoria e grava na camada Bronze com overwriteSchema.
    Retorna métricas de auditoria da gravação.
    """
    df_clean_cols = sanitize_columns(df)

    if source_col_expr is not None:
        df_enriched = df_clean_cols.withColumn("_source_file", source_col_expr)
    else:
        df_enriched = df_clean_cols.withColumn("_source_file", col("_metadata.file_path"))

    df_final = df_enriched.withColumn("_ingestion_timestamp", current_timestamp())

    (
        df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    row_count = spark.table(full_table_name).count()
    col_count = len(df_final.columns)
    return {
        "table": full_table_name,
        "rows": row_count,
        "cols": col_count
    }


def checar_contagem(caminho: str, tabela: str, n_cabecalhos: int = 1) -> None:
    """
    Data Quality Gate da Bronze: compara as linhas do(s) arquivo(s) com as linhas gravadas.
    linhas_arquivo = total de linhas de texto - cabeçalhos (1 por arquivo).
    Falha se a leitura perdeu ou quebrou alguma linha.
    """
    linhas_arquivo = spark.read.text(caminho).count() - n_cabecalhos
    linhas_tabela = spark.table(tabela).count()
    assert linhas_arquivo == linhas_tabela, (
        f"CRÍTICO: {tabela} | arquivo={linhas_arquivo:,} linhas x tabela={linhas_tabela:,} linhas"
    )
    print(f"      -> Completude OK: arquivo = tabela = {linhas_tabela:,} linhas.")

## 3. Validação Rigorosa e Mapeamento Dinâmico da Landing Zone (Pre-flight Check)

In [0]:
expected_items = [
    "unidades_consumidoras.csv",
    "faturas.csv",
    "medidores.csv",
    "ocorrencias_rede.csv",
    "geracao_distribuida.csv",
    "atendimentos.json",
    "inspecoes.xlsx",
    "leituras"
]

# Busca automaticamente em base_dados_1, base_dados_2 ou na raiz do Volume
paths_map = resolve_landing_paths(landing_path, expected_items)

print("✅ Pre-flight check concluído! Todos os 8 itens foram localizados com sucesso:")
for item_name, resolved_path in paths_map.items():
    print(f"   • {item_name:28} -> {resolved_path}")

leituras_files = sorted([
    f for f in os.listdir(paths_map["leituras"])
    if f.lower().endswith(".csv")
])
print(f"\n📂 Arquivos anuais dentro de 'leituras/' ({len(leituras_files)} encontrados): {leituras_files}")

## 4. Ingestão das Tabelas CSV Unitárias (Com Detecção Automática de Delimitador)

In [0]:
csv_tables = {
    "unidades_consumidoras": "unidades_consumidoras.csv",
    "faturas": "faturas.csv",
    "medidores": "medidores.csv",
    "ocorrencias_rede": "ocorrencias_rede.csv",
    "geracao_distribuida": "geracao_distribuida.csv",
}

ingestion_summary = []

for table_suffix, filename in csv_tables.items():
    file_path = paths_map[filename]
    sep = detect_csv_delimiter(file_path)
    target_table = f"{bronze_schema}.tb_{table_suffix}"

    print(f"[CSV] Lendo '{filename}' (delimitador detectado: '{sep}') -> {target_table}")

    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", sep)
        .option("inferSchema", "false")       # 100% String na Bronze para evitar perda/corrupção
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .option("escape", '"')
        .option("multiLine", "false")      # nenhum CSV tem quebra de linha em campo; permite leitura paralela
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(file_path)
    )

    metrics = write_bronze_delta(df_raw, target_table)
    metrics["delimiter"] = sep
    metrics["source_path"] = file_path
    ingestion_summary.append(metrics)
    print(f"      -> Concluído: {metrics['rows']:,} linhas | {metrics['cols']} colunas.")
    checar_contagem(file_path, target_table, n_cabecalhos=1)

## 5. Ingestão das Leituras Mensais Particionadas (`leituras/*.csv`)

In [0]:
leituras_dir = paths_map["leituras"]
sep_leituras = detect_csv_delimiter(leituras_dir)
target_leituras = f"{bronze_schema}.tb_leituras"

print(f"[CSV Particionado] Lendo '{leituras_dir}/*.csv' (delimitador detectado: '{sep_leituras}') -> {target_leituras}")

df_leituras_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("sep", sep_leituras)
    .option("inferSchema", "false")
    .option("encoding", "UTF-8")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "false")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(f"{leituras_dir}/*.csv")
)

metrics_leituras = write_bronze_delta(df_leituras_raw, target_leituras)
metrics_leituras["delimiter"] = sep_leituras
metrics_leituras["source_path"] = f"{leituras_dir}/*.csv"
ingestion_summary.append(metrics_leituras)
print(f"      -> Concluído: {metrics_leituras['rows']:,} linhas | {metrics_leituras['cols']} colunas.")

# 1 cabeçalho por arquivo anual
n_arquivos_leituras = len([f for f in os.listdir(leituras_dir) if f.lower().endswith(".csv")])
checar_contagem(f"{leituras_dir}/*.csv", target_leituras, n_cabecalhos=n_arquivos_leituras)

## 6. Ingestão dos Chamados de Atendimento (`atendimentos.json` Aninhado)

In [0]:
json_path = paths_map["atendimentos.json"]
target_atendimentos = f"{bronze_schema}.tb_atendimentos"

print(f"[JSON Aninhado] Lendo '{json_path}' -> {target_atendimentos}")

df_atendimentos_raw = (
    spark.read.format("json")
    .option("multiLine", "true")
    .option("primitivesAsString", "true")
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .load(json_path)
)

metrics_json = write_bronze_delta(df_atendimentos_raw, target_atendimentos)
metrics_json["delimiter"] = "JSON"
metrics_json["source_path"] = json_path
ingestion_summary.append(metrics_json)
print(f"      -> Concluído: {metrics_json['rows']:,} linhas | {metrics_json['cols']} colunas.")

## 7. Ingestão das Inspeções de Campo (`inspecoes.xlsx` - Múltiplas Abas)

In [0]:
import pandas as pd
from functools import reduce
from pyspark.sql.types import StructType, StructField, StringType

excel_path = paths_map["inspecoes.xlsx"]
excel_file = pd.ExcelFile(excel_path, engine="openpyxl")
target_inspecoes = f"{bronze_schema}.tb_inspecoes"

print(f"[XLSX] Abas identificadas em 'inspecoes.xlsx': {excel_file.sheet_names}")

dfs_abas = []
for sheet_name in excel_file.sheet_names:
    pdf = excel_file.parse(sheet_name=sheet_name, dtype=str)
    pdf = pdf.where(pd.notnull(pdf), None)
    pdf["_source_sheet"] = sheet_name          # aba de origem (metadado, não altera os dados)

    spark_schema = StructType([
        StructField(str(col_name), StringType(), True)
        for col_name in pdf.columns
    ])
    dfs_abas.append(spark.createDataFrame(pdf, schema=spark_schema))
    print(f"      -> Aba '{sheet_name}': {len(pdf):,} linhas lidas.")

# As 3 abas têm as mesmas colunas: empilha tudo numa única tabela Bronze
df_inspecoes = reduce(DataFrame.unionByName, dfs_abas)
total_abas = sum(df.count() for df in dfs_abas)

metrics_xlsx = write_bronze_delta(df_inspecoes, target_inspecoes, source_col_expr=lit(excel_path))
metrics_xlsx["delimiter"] = f"XLSX ({len(dfs_abas)} abas)"
metrics_xlsx["source_path"] = excel_path
ingestion_summary.append(metrics_xlsx)
print(f"      -> Gravado em {target_inspecoes}: {metrics_xlsx['rows']:,} linhas | {metrics_xlsx['cols']} colunas.")

assert metrics_xlsx["rows"] == total_abas, f"CRÍTICO: abas={total_abas:,} x tabela={metrics_xlsx['rows']:,}"
print(f"      -> Completude OK: soma das abas = tabela = {total_abas:,} linhas.")

## 8. Relatório de Auditoria da Camada Bronze (Data Quality Gate)

In [0]:
df_audit = spark.createDataFrame(ingestion_summary).select("table", "delimiter", "rows", "cols", "source_path")
display(df_audit)

# As checagens de completude (arquivo x tabela) já rodaram em cada etapa e fariam o notebook falhar.
# Se a execução chegou até aqui, todas as tabelas Bronze estão completas.
print(f"✅ Camada Bronze concluída: {len(ingestion_summary)} tabelas gravadas e validadas.")